# PynCraft Development and Testing

Notebook for testing changes to the `FruitJuice` (java), `pyncraft`, which 
includes using `rcon` and `mcipc` packages as dependencies.  Also played
around with the `mciwb` and `mcwb` packages.  You probably want to step 
through it and run individual cells, not just run all.

The notebook has been run in a python 3.11.12 virtual environment.  Note that 
you probably don't want to create the environment within this repo.  But if
you're using VSCode, you should be able to access it from whereever it lives.
```
python -m venv venv-pycraft
source venv-pycraft/bin/activate
# note that mciwb installs rcon, mcipc and mcwb as dependencies
pip install \
    html_table_parser \
    ipykernel \
    mciwb \
    "numpy<2.0" \
    "pandas<2.0"
# add pyncraft in editable mode (adjust the path to local pyncraft below)
pip install -e pyncraft
```

A Minecraft Java Edition server is being run from a local docker container.
Connection information is provided below.  The server is being launched using
`docker compose up`.  Ensure that the configuration in the `docker-compose.yml`
is consistent with the values in this notebook.  It is currently using the
`FruitJuice_0.4.0b.jar` (2025-05-23) plugin.  The world SEED is set to 
`9064150133272194`, so if you step through the notebook, you shouldn't 
unexpectedly teleport into a mountain or something.

You may want to adjust video settings for your client to make the world
render faster-- set "Graphics: Fast" is the simplest way.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import math
import os
from pathlib import Path
import re
import time
from typing import Callable, Iterable, Union, Optional

import altair as alt
import numpy as np
import pandas as pd

# from mcipc.rcon.je.client import Client
from mcipc.rcon.je import Biome, Structure, Client
# from mcipc.rcon.types import Vec3
from mciwb.server import MinecraftServer
# from mcwb import Vec3
from mciwb.nbt import parse_nbt

from mcipc.rcon.item import Item
from pyncraft.minecraft import Minecraft 
from pyncraft.vec3 import Vec3


# notebook configuration options
alt.data_transformers.disable_max_rows()

pd.set_option('display.max_columns', None)  # or 1000
pd.set_option('display.max_rows', None)  # or 1000
pd.set_option('display.max_colwidth', None)  # or 199

# block materials (types) from mcipc as a dictionary
ITEMS = {n: m for n, m in Item.__members__.items()}

## Server connection parameters
- These must be consistent with server config in the `docker-compose.yml` file.

In [2]:
# NOTE: you must add the player to the OPS list to use many RCON commands!
player_name = 'Acaimo'
juice_port = 4711

# these only matter if you are using the RCON client
server_name = 'minecraft-server'
server_port = 25565
rcon_pw = 'aspergers-with-cheese'
rcon_port = 25575

# NOTE: you must set the environment variables to values in docker-compose.yml
os.environ['RCON_PASSWORD'] = rcon_pw
os.environ['RCON_PORT'] = str(rcon_port)

## Coord and Vec3D Classes
- Leverage Numpy to avoid reinventing the wheel.
- Treat Vectors as separate objects from Coordinates

In [3]:
class Coord(np.ndarray):
    """
    Implement 3D coordinates as a 3-element (x, y, z) numpy array subclass.

    This avoids having to reinvent the wheel for basic vector operations.
    You can pass in a Vec3, list, tuple, pd.Series or individual x, y, z values.
    They must be numeric (int or float).

    Parameters
    ----------
    x : int, float, Iterable
        The x coordinate or a 3-element iterable containing x, y, z coordinates.
        Positive x is east, negative x is west.
    y : int, float, None
        The y coordinate (vertical position). Positive y is up, negative y is 
        down. Unused if `x` is an iterable.
    z : int, float, None
        The z coordinate. Positive z is south, negative z is north. Unused if 
        `x` is an iterable.
    """

    __int_types__ = (int, np.int32, np.int64)
    __float_types__ = (float, np.float32, np.float64)
    __numeric__ = __int_types__ + __float_types__
    
    def __new__(cls, 
                x: Union[int, float, Iterable], 
                y: Union[int, float, None] = None, 
                z: Union[int, float, None] = None
    ):
        if isinstance(x, Iterable):
            x = [v for i, v in enumerate(x) if i < 4]
            if len(x) != 3:
                raise ValueError(f"Expected 3 coordinates, got {len(x)}: {x}")
            x, y, z = tuple(x)
        else:
            if not isinstance(x, cls.__numeric__):
                raise TypeError(f"x must be int or float, got {type(x)}: {x}")
            if not isinstance(y, cls.__numeric__):
                raise TypeError(f"y must be int or float, got {type(y)}: {y}")
            if not isinstance(z, cls.__numeric__):
                raise TypeError(f"z must be int or float, got {type(z)}: {z}")
        
        obj = np.asarray([x, y, z]).view(cls)
        return obj
    
    # accessor methods convert numpy int/float to Python int/float
    @property
    def x(self):
        return int(self[0]) if isinstance(self[0], self.__int_types__) else float(self[0])

    @x.setter
    def x(self, value):
        self[0] = value

    @property
    def y(self):
        return int(self[1]) if isinstance(self[1], self.__int_types__) else float(self[1])

    @y.setter
    def y(self, value):
        self[1] = value

    @property
    def z(self):
        return int(self[2]) if isinstance(self[2], self.__int_types__) else float(self[2])

    @z.setter
    def z(self, value):
        self[2] = value

    
    def adjust(self, axis: str, delta: Union[int, float]) -> 'Coord':
        """
        Copy Coord and adjust one axis (x, y or z) by delta
        """
        axis = {'x': 0, 'y': 1, 'z': 2}.get(axis.lower())
        coord = self.copy()
        coord[axis] += delta
        return coord


    def block(self, as_int: bool = True) -> 'Coord':
        """
        Get the whole-number block coordinates as Coord object By Flooring.

        If as_int is True, returns the block coordinate as an integer.
        Otherwise, float.
        """
        block = np.floor(self)
        if as_int:
            block = block.astype(int)
        return block
    
    
    def midblock(self) -> 'Coord':
        return self.block() + Coord(0.5, 0.0, 0.5)
    

    def coplane(self, c2: 'Coord') -> bool:
        """
        Check if second coordinate is in the same plane.
        """

        return Vec3D(self, c2).coplane()
    

    def direction(self, c2: 'Coord') -> 'Vec3D':
        """
        Get the unit vector from c1 to c2 (normalized to length 1).
        """
        return Vec3D(self, c2).direction()
    
    
    def cardinal_direction(self, c2: 'Coord', *args, **kwargs) -> Union[str, 'Coord']:
        """
        Get the Cardinal Direction of a Second Coordinate.

        See help for Vec3D.cardinal_direction() for details.
        """
        return Vec3D(self, c2).cardinal_direction(*args, **kwargs)
    

    def distance(self, c2: 'Coord') -> float:
        """
        Calculate the distance between two coordinates.
        """
        return float(np.sqrt(((self - c2) ** 2).sum()))
        


class Vec3D(Coord):
    """
    A 3D vector class that takes one or two Coord objects as input.
    """
    
    def __init__(self, c1: Coord, c2: Coord = None):

        v = c1 if c2 is None else c2 - c1
        
        self.x = v.x
        self.y = v.y
        self.z = v.z
    

    def is_unit(self):
        """Test if the vector is a unit vector (normalized to length 1)."""
        return np.isclose(np.linalg.norm(self), 1.0)
    

    def direction(self) -> Coord:
        """
        Get the unit vector from c1 to c2 (normalized to length 1).
        """
        return (self) / np.linalg.norm(self)
    

    def coplane(self, by_block: bool = True) -> bool:
        """
        Check if second coordinate is in the same plane.
        """

        if by_block:
            pass

        plane = ''
        if np.isclose(self.x, 0.0):
            plane += 'x'
        if np.isclose(self.y, 0.0):
            plane += 'y'
        if np.isclose(self.z, 0.0):
            plane += 'z'

        return plane
    
    
    def cardinal_direction(
            self, 
            returns: str = 'string',
            compass_points: int = 4
        ) -> Union[str, int, Coord]:
        """
        Get the Cardinal Direction Of The Vector In The XY Plane.

        Uses the x and z components only, and returns the closest direction
        based on the number fo compass points.  Direction may be returned 
        as a string ("north"), minecraft rotation value (0-15), degrees (0-360),
        radians (0-2pi), or a Vec3D/Coord unit vector with y==0.

        Note that when there is a tie (like x=1 and z=-1 with 4 compass points),
        I'm not doing anything specific to resolve (north or east in this case).
        Could return 'indeteminate', None/np.nan or raise an exception.  Also
        leaving it up to the user whether to convert Coords to integer.

        Parameters
        ----------
        returns : str
            The type of return value. Can be 'string', 'rotation', 'coord', or 
            'degrees'. Default is 'string'.
        compass_points : int
            The number of compass points to use for the direction. 
            Default is 4 (north, east, south, west). If set to 8, it will also
            return northeast, southeast, southwest, and northwest.  If set
            to 16, it will return all 16 compass points ('northnortheast', 
            'northeast', 'eastnortheast', 'east', etc).  
        
        Returns
        -------
        Union[str, int, Coord]:
            If returns is string: 'north', 'south', 'east', 'west',.
        """
        ok = {'string', 'rotation', 'coord', 'degrees', 'radians'} 
        invalid = {returns} - ok
        if invalid:
            raise ValueError(
                'Invalid returns argument "{returns}"; must be one of:\n   '
                + ', '.join(list(ok)))
        
        if self.x == 0 and self.z == 0:
            raise ValueError('Vector point straight up or down; indeterminate.')

        # drop the y component and normalize the xz vector (z -> y)
        xz = np.delete(self, 1)
        xz = xz / np.linalg.norm(xz)

        # atan2 returns angle from x-axis (east), so swap arguments and invert z
        # calculate fraction of a turn [0, 1) from North = 0 degrees
        frac360 = (np.arctan2(xz.x, -xz.y) / (np.pi * 2)) % 1.0

        bearings = ['north', 'northnortheast', 'northeast', 'eastnortheast', 
                    'east', 'eastsoutheast', 'southeast', 'southsoutheast', 
                    'south', 'southsouthwest', 'southwest', 'westsouthwest',
                    'west', 'westnorthwest', 'northwest', 'northnorthwest',
                    'north']

        if compass_points > 0:
            # convert to a rotation value, rounded to the nearest compass point
            rotation = int(
                np.round(frac360 * compass_points) * 16 / compass_points)

            if returns == 'rotation':
                return rotation
            elif returns == 'degrees':
                return rotation * 360 / 16
            elif returns == 'string':
                return bearings[rotation]
            else: 
                radians = rotation * 2 * np.pi / 16
                if returns == 'radians':
                    return radians
                else:
                    # rounded unit vector in the xz plane
                    coord = Coord(np.sin(radians), 0, -np.cos(radians))
                    if compass_points == 4:
                        coord = coord.astype(int)
                    return coord
        elif returns in ('string', 'rotation'):
            raise ValueError(
                f'Must set compass_points > 0 for returns "string" or "rotation"')
        else:
            # return results without rounding
            if returns == 'degrees':
                return frac360 * 360
            else:
                if returns == 'radians':
                    return radians
                else:
                    return Coord(np.sin(radians), 0,-np.cos(radians))
        

In [4]:
# some testing of Vec3.cardinal_direction() method
r = 10000.0
angles = np.linspace(0, 2 * np.pi / 4, 12, endpoint=False)
df = pd.DataFrame({
    'angle': 180 * angles / np.pi,
    'x': np.round(r * np.sin(angles)),
    'y': [0] * len(angles),
    'z': np.round(-r * np.cos(angles)),
    'string': [''] * len(angles),
    'rotation': [0] * len(angles),
    'degrees': [0.0] * len(angles),
    'radians': [0.0] * len(angles),
    'coord.x': [0.0] * len(angles),
    'coord.y': [0.0] * len(angles),
    'coord.z': [0.0] * len(angles),
})

cp = 4
for i in df.index:
    # v = Vec3D(Coord(df.x[i], df.y[i], df.z[i]))
    v = Vec3D(Coord(df.loc[i, ['x', 'y', 'z']]))
    df.loc[i, 'string'] = v.cardinal_direction(returns='string', compass_points=cp)
    df.loc[i, 'rotation'] = v.cardinal_direction(returns='rotation', compass_points=cp)
    df.loc[i, 'degrees'] = v.cardinal_direction(returns='degrees', compass_points=cp)
    df.loc[i, 'radians'] = v.cardinal_direction(returns='radians', compass_points=cp)
    df.loc[i, [f'coord.{x}' for x in 'xyz']] = \
        v.cardinal_direction(returns='coord', compass_points=cp).round(3)

display(df)

,angle,x,y,z,string,rotation,degrees,radians,coord.x,coord.y,coord.z
0,0.0,0.0,0,-10000.0,north,0,0.0,0.000000,0.0,0.0,-1.0
1,7.5,1305.0,0,-9914.0,north,0,0.0,0.000000,0.0,0.0,-1.0
2,15.0,2588.0,0,-9659.0,north,0,0.0,0.000000,0.0,0.0,-1.0
3,22.5,3827.0,0,-9239.0,north,0,0.0,0.000000,0.0,0.0,-1.0
4,30.0,5000.0,0,-8660.0,north,0,0.0,0.000000,0.0,0.0,-1.0
5,37.5,6088.0,0,-7934.0,north,0,0.0,0.000000,0.0,0.0,-1.0
6,45.0,7071.0,0,-7071.0,north,0,0.0,0.000000,0.0,0.0,-1.0
7,52.5,7934.0,0,-6088.0,east,4,90.0,1.570796,1.0,0.0,0.0
8,60.0,8660.0,0,-5000.0,east,4,90.0,1.570796,1.0,0.0,0.0
9,67.5,9239.0,0,-3827.0,east,4,90.0,1.570796,1.0,0.0,0.0


## Connect to Minecraft Server with FruitJuice plugin

#### note that player must be connected to the server to connect

In [5]:
mc = Minecraft.create('localhost', port=juice_port, playerName=player_name)

 ('Running Python version: 3.11.12 (main, Apr 21 2025, 17:46:22) [Clang 17.0.0 (clang-1700.0.13.3)]',) 
 ('get Acaimo playerid=38',) 


In [6]:
# assuming seed is `9064150133272194`
print(mc.runCommands('seed'))

spawn_point = (-4.5, 109, 5.5)

['Seed: [9064150133272194]']


In [7]:
def object_info(obj):
    """list attributes and methods of an object"""
    print('attributes:')
    print('\n'.join(
        [x for x in dir(obj) if not x.startswith('_') 
         and not isinstance(getattr(obj, x), Callable)]))

    print('\nmethods:')
    print('\n'.join(
        [x for x in dir(obj) if not x.startswith('_') 
         and isinstance(getattr(obj, x), Callable)]))
    

# check out Minecraft object's attributes and methods
object_info(mc)

attributes:
camera
cmdplayer
conn
entity
events
player
playerId
settings

methods:
create
createExplosion
getBlock
getBlockWithData
getBlocks
getHeight
getPlayerEntityId
getPlayerEntityIds
postToChat
restoreCheckpoint
runCommands
saveCheckpoint
setBlock
setBlocks
setSign
setWallSign
setting
spawnEntity


In [8]:
# a command from the FruitJuice plugin
print(f'world height at x=50, z=-160: {mc.getHeight(50, -160)}')

world height at x=50, z=-160: 62


### Test whether using `runCommands()` via the RCON protocol works

In [9]:
def teleport(pos: Iterable):
    """
    Teleport the player to a given position.

    Note: mc.player.setTilePos() doesn't seem to work correctly.
    
    Parameters
    ----------
    pos : Iterable
        An iterable of 3 floats representing the x, y, and z coordinates.
    """
    
    pos = Coord(pos)
    here = Coord(mc.player.getTilePos()).midblock()

    there = [f'{float(x)}' for x in pos]
    there = ' '.join(there)
    result = mc.runCommands(f'tp {player_name} {there}')[0]

    if result.startswith('Teleported'):
        msg = f'Teleported {player_name} from {here} to {pos}'
        print(msg.replace('  ', ' ').replace('[ ', '['))
    return 

In [10]:
teleport(spawn_point)

Teleported Acaimo from [7.5 100.  -1.5] to [-4.5 109.  5.5]


In [11]:
# stop sending command output to the player's chat
mc.runCommands('gamerule sendCommandFeedback false')

['Gamerule sendCommandFeedback is now set to: false']

In [12]:
def dawn(self):
    self.runCommands(['time set 2350s', 'weather clear'])
  
dawn(mc)

In [14]:
# cycle through the hours of the day, 1 hr per second
results = mc.runCommands(
    [f'time set {1000 * ((x + 6) % 24)}' for x in range(0, 24)], 1)

### Test my extensions of the Fruit Juice plugins
- `world.getBlockData`: returns BlockState as a string with material and 
  all other attributes (which vary by material).

In [15]:
teleport(Coord(11.5, 91, -10.5))
pos = mc.player.getTilePos()

blockdata = []
for dy in range(1, -12, -1):
    # pos.y -= 1
    blockdata.append(mc.getBlockWithData(pos.x, pos.y + dy, pos.z, parse=False))


blockdata = pd.DataFrame(blockdata)
blockdata#.fillna('')

# mc.getBlockWithData(mc, pos)

Teleported Acaimo from [-4.5 109.  5.5] to [11.5 91. -10.5]


,x,y,z,material,state
0,11.0,92.0,-11.0,AIR,
1,11.0,91.0,-11.0,PINK_PETALS,"facing=south,flower_amount=1"
2,11.0,90.0,-11.0,GRASS_BLOCK,snowy=false
3,11.0,89.0,-11.0,DIRT,
4,11.0,88.0,-11.0,DIRT,
5,11.0,87.0,-11.0,DIORITE,
6,11.0,86.0,-11.0,DIORITE,
7,11.0,85.0,-11.0,DIRT,
8,11.0,84.0,-11.0,DIRT,
9,11.0,83.0,-11.0,DIRT,


In [16]:
def getBlocksDf(
        self, 
        x1: int, 
        y1: int, 
        z1: int, 
        x2: int, 
        y2: int, 
        z2: int,
        with_state: bool = False
    ) -> pd.DataFrame:
    """
    Get materials of a block cuboid as a Pandas DataFrame, specified by corners.

    Parameters
    ----------
    x1, y1, z1 : int
        Coordinates of one corner of the cuboid.
    x2, y2, z2 : int
        Coordinates of the opposite corner of the cuboid (inclusive).
    with_state : bool, optional
        If True, also get the block state as a string (default is False).
        Note that this will slow down the function significantly.

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns 'x', 'y', 'z', and 'material'.
        The material is a string representation of the block.
    """

    start, stop = zip(sorted([x2, x1]), sorted([y1, y2]), sorted([z2, z1]))
    blocks = self.conn.sendReceive(b"world.getBlocks", *start, *stop).split(',')
    
    x = np.arange(start[0], stop[0]+1)
    y = np.arange(start[1], stop[1]+1)
    z = np.arange(start[2], stop[2]+1)
    x1, y1, z1 = np.meshgrid(x, y, z)
    
    df = pd.DataFrame({
        'x': x1.flatten(), 
        'y': y1.flatten(), 
        'z': z1.flatten(),
        'material': blocks})
    
    if with_state:
        df['state'] = ''
        df = df.set_index(['x', 'y', 'z']).sort_index()
        # get the block data for each block
        for v3 in df.index:
            if df.loc[v3, 'material'] != 'AIR':
                state = self.getBlockWithData(*v3, parse=False)['state']
                df.loc[v3, 'state'] = state
                
        df.reset_index(inplace=True)

    return df


# add this to the Minecraft object (should add it to the class)
mc.getBlocksDf = getBlocksDf

In [17]:
# use juice getBlocks to get materials of a cuboid as an array of strings
pos = Coord(mc.player.getTilePos()).block()
print(pos)
here = (pos.x-2, pos.y, pos.z-2)
there = (pos.x+2, pos.y+1, pos.z+2)

mc.getBlocks(*here, *there)

[ 11  91 -11]


[[['GRASS_BLOCK', 'GRASS_BLOCK', 'GRASS_BLOCK', 'GRASS_BLOCK', 'GRASS_BLOCK'],
  ['GRASS_BLOCK', 'GRASS_BLOCK', 'GRASS_BLOCK', 'GRASS_BLOCK', 'GRASS_BLOCK']],
 [['OXEYE_DAISY', 'AIR', 'PINK_PETALS', 'SHORT_GRASS', 'AIR'],
  ['AIR', 'AIR', 'DANDELION', 'DANDELION', 'AIR']],
 [['AIR', 'AIR', 'DANDELION', 'PINK_PETALS', 'SHORT_GRASS'],
  ['SHORT_GRASS', 'AIR', 'PINK_PETALS', 'AIR', 'PINK_PETALS']],
 [['AIR', 'AIR', 'PINK_PETALS', 'PINK_PETALS', 'PINK_PETALS'],
  ['AIR', 'AIR', 'AIR', 'AIR', 'AIR']],
 [['AIR', 'AIR', 'AIR', 'AIR', 'AIR'], ['AIR', 'AIR', 'AIR', 'AIR', 'AIR']]]

In [18]:
print(*here, *there)
cuboid = getBlocksDf(mc, *here, *there, with_state=True)

# # sanity check - this is slow, but one-off to check that coords are correct
# for row in cuboid.itertuples(): 
#     checktype = mc.getBlock(row.x, row.y, row.z)
#     assert checktype == row.material, \
#         f"Block type mismatch at ({row.x}, {row.y}, {row.z}): {row.material} != {checktype}"

display(cuboid)#[cuboid['state'].gt('')])
print(len(cuboid))

9 91 -13 13 92 -9


,x,y,z,material,state
0,9,91,-13,GRASS_BLOCK,snowy=false
1,9,91,-12,GRASS_BLOCK,snowy=false
2,9,91,-11,GRASS_BLOCK,snowy=false
3,9,91,-10,GRASS_BLOCK,snowy=false
4,9,91,-9,GRASS_BLOCK,snowy=false
5,9,92,-13,SHORT_GRASS,
6,9,92,-12,AIR,
7,9,92,-11,PINK_PETALS,"facing=south,flower_amount=2"
8,9,92,-10,AIR,
9,9,92,-9,PINK_PETALS,"facing=west,flower_amount=3"


50


### function to render many blocks, with specified state

In [19]:
def place_blocks_with_state(self, df: pd.DataFrame):

    df = df.copy()
    for col in ['x', 'y', 'z']:
        if df.dtypes[col].__str__() == 'float64':
            df[col] = df[col].round().astype(int)

    missing = list({'x', 'y', 'z', 'material'} - set(df.columns))
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    
    df['material'] = 'minecraft:' + df['material'].str.lower()
    # df['material'] = df['material'].map(ITEMS)
    # if df['material'].isnull().any():
    #     raise ValueError(f"Unknown material: {df['material'].isnull().sum()}")
    
    if 'state' in df.columns:
        df['state'] = np.where(
            df['state'].gt(''), '[' + df['state'].copy() + ']', '')
        df['material'] = df['material'] + df['state']

    commands = df[['x', 'y', 'z', 'material']].apply(
        lambda r: ' '.join(r.astype(str)), axis=1)
    commands = [f'setblock {x}' for x in commands]

    # should maybe chunk the commands and pause between them to avoid overload
    results = self.runCommands(commands, 0.1)
    df['success'] = [x.startswith('Changed the block at') for x in results]

    return df

In [20]:
# copy the cuboid above to a new location
teleport(Coord(-189.5, 138, -271.5))
new_pos = Coord(mc.player.getTilePos()).block()

move = new_pos - pos
move

Teleported Acaimo from [11.5 91. -10.5] to [-189.5 138. -271.5]


Coord([-201,   47, -261])

In [21]:
moved = cuboid.copy()
moved['x'] += move.x
moved['y'] += move.y
moved['z'] += move.z

results = place_blocks_with_state(mc, moved)
display(results)

,x,y,z,material,state,success
0,-192,138,-274,minecraft:grass_block[snowy=false],[snowy=false],True
1,-192,138,-273,minecraft:grass_block[snowy=false],[snowy=false],True
2,-192,138,-272,minecraft:grass_block[snowy=false],[snowy=false],True
3,-192,138,-271,minecraft:grass_block[snowy=false],[snowy=false],True
4,-192,138,-270,minecraft:grass_block[snowy=false],[snowy=false],True
5,-192,139,-274,minecraft:short_grass,,True
6,-192,139,-273,minecraft:air,,False
7,-192,139,-272,"minecraft:pink_petals[facing=south,flower_amount=2]","[facing=south,flower_amount=2]",True
8,-192,139,-271,minecraft:air,,False
9,-192,139,-270,"minecraft:pink_petals[facing=west,flower_amount=3]","[facing=west,flower_amount=3]",True


### Some general building functions, to be added to `Builder` or `Unit` class


In [22]:
# just for backwards compatibility with older version of the subway code
blocktypes = {
    0: 'AIR',
    1: 'STONE',
    2: 'GRASS_BLOCK', #
    3: 'DIRT',
    5: 'DARK_OAK_WOOD',  #
    9: 'BLUE_ICE', 
    20: 'GLASS',
    27: 'POWERED_RAIL',
    76: 'REDSTONE_TORCH',
    89: 'GLOWSTONE',
    98: 'STONE_BRICKS', #
    107: 'BIRCH_FENCE_GATE' #
}


def np_rep(
        x: np.array, 
        reps: int=1, 
        each: bool=False, 
        length: int=0
    ) -> np.array:
    """
    Replicate the values in x, similar to R's rep() and rep_len() functions.
    
    Parameters
    ----------
    x : numpy array
        The array to be replicated.
    reps : int, optional
        The number of times to replicate the values in x. Default is 1.
    each : bool, optional
        If True, each element of x is repeated reps times before the next
        element. Default is False.
    length : int, optional
        The desired length of the output array. If greater than 0, this 
        overrides the reps argument. Default is 0.

    Returns
    -------
    numpy array
        The replicated array, with length equal to len(x) * reps or the 
        length if specified.
    """

    if length > 0:
        reps = np.int(np.ceil(length / x.size))
    x = np.repeat(x, reps)
    if not each:
        x = x.reshape(-1, reps).T.ravel()
    if length > 0:
        x = x[0:length]
    return x


# def circle_of_blocks(radius, blocktype = 1):
#     '''
#     Create a DataFrame of block positions in a circle with a given radius.
#     '''
#     radius += 0.25
#     dTheta = math.asin(0.25 / radius)
#     theta = np.array(np.arange(0, 2 * math.pi, dTheta))
#     df = pd.DataFrame({
#         'theta': theta,
#         'x': np.round(radius * np.cos(theta)),
#         'y': np.round(radius * np.sin(theta)),
#         'blocktype': blocktype
#     })
#     df['x'] = df['x'].astype(int)
#     df['y'] = df['y'].astype(int)
#     return df.groupby(['x', 'y', 'blocktype']).mean().reset_index()


def extruder(
        df: pd.DataFrame, 
        startpos: tuple, 
        dir: str="y", 
        distance: int=1,
        rcon: bool=False,
        pause: float=0,
    ) -> int:
    """
    Given A DataFrame of 2D coordinates, extrude in 3D From Start Position

    Note: this is a stupid implementation.

    Parameters
    ----------
    df : pd.DataFrame
        A DataFrame of 2D coordinates with columns 'x', 'y', and 'blocktype'.
        Positions will be instantiated relative to the start position.
    startpos : tuple
        A tuple of 3 integers representing the starting position in 3D space.
        The coordinates are in the order (x, y, z).
    dir : str, optional
        The direction to extrude in. Can be 'x', 'y', or 'z'. Default is 'y'.
        Positive x is east, positive y is up, and positive z is south.
    distance : int, optional
        Distance to extrude in the direction specified, by default 1
    rcon: bool, optional
        If True, use the RCON client to place blocks, otherwise use the 
        FruitJuice setBlock() method. Default is False.  RCON can set block
        state, but seems to have more issues with server overload.  Probably
        should use RCON only for blocks with non-default states?
    pause: float, optional
        Pause in seconds between placing each block, by default 0.
        This is useful to avoid overloading the server with too many commands
        at once, especially when using RCON.

    Returns
    -------
    int
        The total number of blocks placed.
    """

    n = 0
    if rcon:
        # use the RCON client to place blocks; seems to have more lag issues
        if(dir[0]=='-'):
            offset = -1
            dir = dir[1]
        else:
            offset = 1 

        segment = df.copy().rename(columns={'blocktype': 'material'})
        segment['x'] = startpos[0]
        segment['y'] = startpos[1]
        segment['z'] = startpos[2]
        segment['stats'] = ''
        if dir == "x":
            segment['y'] += df['y']
            segment['z'] += df['x']
        elif dir == "y":
            segment['x'] += df['x']
            segment['z'] += df['y']
        elif dir == "z":
            segment['x'] += df['x']
            segment['y'] += df['y']

        for i in range(distance):
            segment[dir] += offset
            results = place_blocks_with_state(mc, segment)
            n += len(segment)

    else:
        # use the FruitJuice setBlock() method
        if(dir[0]=='-'):
            offsets = range(0, -distance, -1)
            dir = dir[1]
        else:
            offsets = range(0, distance)

        for offset in offsets:
            time.sleep(pause)
            for index, row in df.iterrows():
                if dir == "x":
                    d = [offset, row.y, row.x]
                elif dir == "y":
                    d = [row.x, offset, row.y]
                elif dir == "z":
                    d = [row.x, row.y, offset]
                mc.setBlock(
                    startpos[0] + d[0], 
                    startpos[1] + d[1], 
                    startpos[2] + d[2],
                    row.blocktype)
                n += 1

    return n


In [23]:
def circle_of_blocks(
        d: int, 
        cutoff: float = 0.45, 
    ) -> pd.DataFrame:
    """
    Generate a DataFrame of block positions in a circle with a given radius.

    Uses a heuristic to determine positions of blocks that are fairly round.
    May have gaps in the perimeter for very large diameters (like >100?)

    The center of the circle is positioned at x=0, y=0. Note that for even 
    diameters, the block positions are half-block centered.

    Parameters
    ----------
    d : int
        The diameter of the circle; integer > 2
    cutoff : float, optional
        Cutoff that impacts width and roundness. Default 0.45 gives good 
        results for most diameters.  Larger values exclude blocks with 
        a smaller proportion of the circle's perimeter inside them.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns 'x', 'y', 'n', and 'theta'.  Theta is in radians.
    """

    if d < 2:
        raise ValueError(f"Diameter must be greater than 2, got {d}")
    
    fun = np.floor if d % 2 == 0 else np.round
    theta = np.linspace(0.0, 2 * np.pi, 10000, endpoint=False)
    r = (d / 2) - 0.5
    df = pd.DataFrame({
        'x': fun(r * np.cos(theta)).astype(int),
        'y': fun(r * np.sin(theta)).astype(int),
    })
    df = df.groupby(['x', 'y']).size().reset_index().rename(columns={0: 'n'})
    df['n'] /= df['n'].max() 
    df = df[df['n'].gt(cutoff)]
    if d % 2 == 0:
        df['x'] += 0.5
        df['y'] += 0.5
    df['theta'] = np.arctan2(df['y'], df['x'])
    df = df.sort_values('theta').reset_index()

    return df

In [24]:
# show output of circle_of_blocks() for various diameters
charts = []
for i, d in enumerate(range(3, 27)):
    if i % 6 == 0:
        row = []

    r = np.ceil(d / 2.0)
    circle_df = circle_of_blocks(d, 0.45)  

    if d in (5, 6):
        circle_df['n'] = (10 * circle_df['n'].round(1)).astype(int).astype(str)
        display(circle_df.pivot(index='y', columns='x', values='n').fillna(''))

    circle_df['n'] = 1.0
    chart = alt.Chart(circle_df).mark_rect().encode(
        x=alt.X('x:O', axis=alt.Axis(labels=False, title='')),
        y=alt.Y('y:O', axis=alt.Axis(labels=False, title='')),
        color=alt.Color('n:N', 
            scale=alt.Scale(scheme='greys'), 
            bin=alt.Bin(maxbins=10),
        ),
        tooltip = ['n', 'theta:Q'],
    ).properties(
        width=100, 
        height=100,
        title=f'{d}',
    )
    row.append(chart)
    if i % 6 == 5:
        charts.append(
            alt.hconcat(*row).resolve_scale(x='independent', y='independent'))
        
        
charts = alt.vconcat(*charts).resolve_scale(x='independent', y='independent')
display(charts)



x,-2,-1,0,1,2
y,,,,,
-2,,9,10,9,
-1,9,,,,9
0,10,,,,10
1,9,,,,9
2,,9,10,9,


x,-2.5,-1.5,-0.5,0.5,1.5,2.5
y,,,,,,
-2.5,,6,10,10,6,
-1.5,6,7,,,7,6
-0.5,10,,,,,10
0.5,10,,,,,10
1.5,6,7,,,7,6
2.5,,6,10,10,6,


alt.VConcatChart(...)

In [25]:
def display_block_types(distance: int = 2):
   """
   Generate each block type with a sign indicating ID and name on top
   
   Blocks are placed in a line in front of the player, with signs above,
   facing the player.  Note that this is using the Juice plugin's
   `setBlock` command.
   """

   pos = Coord(*mc.player.getTilePos()).astype(int)
   pointing = Vec3D(mc.player.getDirection())
   
   facing = pointing.cardinal_direction().upper()
   pointing = pointing.cardinal_direction('coord')

   # print(f'Player position: {pos}, pointing: {pointing} ({facing})')

   # place the blocks in a line in front of the player
   start = pos + distance * pointing

   # center left to right from their perspective
   if pointing.x == 0:
      start.x += pointing.z * -len(blocktypes)
      translate = Coord(pointing.z, 0, 0)
   else:
      start.z += pointing.x * -len(blocktypes)
      translate = Coord(0, 0, pointing.x)

   # print(start)
   flip_face = {
      'NORTH': 'SOUTH', 'SOUTH': 'NORTH', 'EAST': 'WEST', 'WEST': 'EAST'
   }
   for i, k in enumerate(blocktypes.keys()):
      here = start + 2 * i * translate
      # print(f'Placing {here} ({blocktypes[k]})')
      mc.setBlock(*here, blocktypes[k], 'NORTH')
      here.y += 1
      mc.setSign(*here, 'BIRCH_SIGN', flip_face[facing], 
                 f'{k} =', *(blocktypes[k]).split('_'))
      

# field with big flat area of grass
teleport(Coord(-29, 70, -185))
display_block_types(distance=3)

Teleported Acaimo from [-189.5 138. -271.5] to [-29  70 -185]


In [26]:
# test the circle_of_blocks + extruder functions for making cylinders
pos = Coord(17.0, 91, -13.0)  # another flattish area
teleport(pos)

pos = Coord(mc.player.getTilePos()).midblock()
for ring in range(3):
    df = circle_of_blocks(7 + ring * 3)
    df['blocktype'] = 'GLASS'
    extruder(df, pos, "y", int(7 - (ring * 2)))
    time.sleep(0.5)


Teleported Acaimo from [-28.5  70. -184.5] to [17. 91. -13.]


In [27]:
# setDirection doesn't work-- always sets direction to 1,0,0
pointing = Vec3D(mc.player.getDirection())
display(pointing.round(3))
display(pointing.cardinal_direction())

mc.player.setDirection(0.0, 0.0, 1.0)  # FAILS!
pointing = mc.player.getDirection()
[round(float(x)) for x in pointing]

array([-0.375, -0.571,  0.731])

'south'

[1, 0, 0]

In [34]:
def set_in_front(mc, blocktype, distance=2):
    """
    Get player's current position and direction; set block in front of player.
    """
    pointing = Vec3D(mc.player.getDirection())
    facing = pointing.cardinal_direction()
    pointing = pointing.cardinal_direction('coord')
    pos = Coord(mc.player.getTilePos()).astype(int)

    print(f'pointing: {pointing} ({facing})')
    print(f'position: {pos}')

    pos += pointing * distance
    print(f'  target: {pos}')
    mc.setBlock(*pos, blocktype)



def whats_that(mc):
    """
    Get material of block the player is looking at (approximately).

    This doesn't work that well-- not how it should handle partially 
    transparent blocks like short grass or slabs, and camera is offset by 0.05 
    blocks in the direction the player is looking.  But can be used to 
    judge distances to distant blocks.
    """

    pos = Coord(mc.player.getPos())
    pos.y += 1.62  # camera height
    pointing = Vec3D(mc.player.getDirection())

    for d in range(255):
        that = pos + d * pointing
        block = mc.getBlockWithData(*that)
        if block['material'] not in ['AIR', 'WATER']:
            block.update({'distance': d})
            break
        
    return block


# set_in_front(mc, 'WHITE_BED')
# set_in_front(mc, 'WHITE_BED', 3)
whats_that(mc)


{'x': 61.83279883219388,
 'y': 122.7580305872482,
 'z': 554.9161118223144,
 'material': 'CHERRY_LEAVES',
 'distance': 107,
 'persistent': 'false',
 'waterlogged': 'false'}

### Functions to create a "Subway" Section on Support Columns

In [30]:
def subway_section(radius: int) -> pd.DataFrame:
    """
    Creates A Cross-Section of a Subway Tube As A DataFrame

    Parameters
    ----------
    radius : int
        Radius of the tube section in blocks.  Optimized for 4.

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns 'x', 'y', 'radius', 'theta', and 'blocktype'.
    """

    section = np.arange(-(radius + 1), radius + 2)
    section = pd.DataFrame({
        'x': np_rep(section, section.size),
        'y': np_rep(section, section.size, each=section.size)
    })
    section['radius'] = np.sqrt(section.x**2 + section.y**2)
    section['theta'] = np.arctan(section.y / section.x) * 180 / math.pi
    section['blocktype'] = np_rep(0, section.shape[0])
    
    # build up the platform
    section.loc[(section.x==0) & (section.y==-2), 'blocktype'] = 98 # stone brick
    section.loc[(section.x==0) & (section.y==-1), 'blocktype'] = 27 # powered rail
    section.loc[(np.abs(section.x)==1) & (section.y==-3), 'blocktype'] = 89 # glowstone
    section.loc[(np.abs(section.x)==1) & (section.y==-2), 'blocktype'] = 2 # grass
    # set the tube to glass
    section.loc[round(section.radius) == radius, 'blocktype'] = 20
    # wooden supports
    section.loc[(round(section.radius) == radius)
        & (np.abs((section.theta % 60) - 30) < 4), 'blocktype'] = 5
    section.loc[(section.x==0) & (section.y==-3), 'blocktype'] = 76 # redstone torch

    section.loc[(np.abs(section.x)==2) & (section.y==-2), 'blocktype'] = 9 # water
    # external bottom supports
    section.loc[(np.abs(section.x)==3) & (section.y==-4), 'blocktype'] = 98 # stone brick
    section.loc[(np.abs(section.x)==4) & (section.y==-4), 'blocktype'] = 98 # stone brick
    section.loc[(np.abs(section.x)==1) & (section.y==-5), 'blocktype'] = 98 # stone brick
    # drop surrounding part
    section = section.loc[(np.round(section.radius) <= radius) | (section.blocktype!=0)]
    section['blocktype'] = section['blocktype'].map(blocktypes)
    # switch out the BLUE_ICE for WATER (lights up due to glowstone)
    section.loc[section.blocktype == 'BLUE_ICE', 'blocktype'] = 'WATER'
    
    return section


def support(across: int=4, blocktype: int=1) -> pd.DataFrame:
    """
    Square Section of a Subway Tube Support, Dimension 2 * across + 1 

    Parameters
    ----------
    across : int, optional
        Half (width -1), by default 4
    blocktype : int, optional
        Material as a number, by default 1

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns 'x', 'y', and 'blocktype' (as string).
    """

    section = np.arange(-across, across + 1)
    section = pd.DataFrame({
        'x': np_rep(section, section.size),
        'y': np_rep(section, section.size, each=section.size),
        'blocktype': np_rep(blocktype, section.size ** 2)
    })
    section.loc[(np.abs(section.x) < 3) & (section.y == 0), 'blocktype'] = 0
    section.loc[(section.x == 0) & (np.abs(section.y) < 3), 'blocktype'] = 0
    section = section.loc[section.blocktype != 0]
    section['blocktype'] = section['blocktype'].map(blocktypes)
    
    return section


def arch_df(section_length: int, material: str, dir: str='z') -> pd.DataFrame:

    arch = circle_of_blocks(section_length - 5)
    arch['blocktype'] = material
    arch.drop('theta', axis=1, inplace=True)
    arch = arch[arch['y'].ge(0)]
    arch['y'] = arch['y'] - arch['y'].max()
    # arch = arch[arch['x'].lt((section_length - 2) / 2)]

    sign = -1 if dir[0] == '-' else 1
    arch['x'] = arch['x'] + sign * section_length / 2

    return arch

In [31]:
def build_subway_line(
        here: Coord,
        there: Coord,
        section_lengths: int = 24,
        radius: int = 4,
        column_radius: int = 4,
        max_support_height: int = 40,
        delay_factor: float = 2.0,
        calcs_only: bool = False
    ) -> int:
    """
    Build A Subway Line Between Two Coordinates

    The subway line must currently be on a straight path between two points 
    in the same plane (xy or yz).  The line is built in sections of a given
    length, with any extra blocks placed at the ends, as on and off-ramps.
    There are junctions between sections with two rings of stone and a gap,
    and a support column under each junction.  Under each section (but not
    the ramps) there is a support arch.  At each end of the powewered rail
    way, there is an unpowered rail and a stone block, to keep the minecart
    from going off the ends of the line.

    Parameters
    ----------
    here : Coord
        Starting coordinate of the subway line, as a Coord object.
    there : Coord
        Ending coordinate of the subway line, as a Coord object.
    section_lengths : int, optional
        Lengths of sections in the middle, by default 24.
    radius : int, optional
        Radius of the tube, by default 4 (build has been optimized for 4).
    column_radius : int, optional
        Specifies column size, where it will be twice this value + 1 at the
        top.  By default 4; should be at least 3.
    max_support_height : int, optional
        Maximum distance support columns are expected to be above the surface
        below, by default 40.  Currently the colums will extend this far down
        regardless of the height of the surface below (stupid).
    delay_factor : float, optional
        Controls how long to wait between placing chunks of the build.
        Increase this if the server is getting overwhelmed, by default 2.0
    calcs_only : bool, optional
        If True, just print out info about what will be built. By default False.

    Returns
    -------
    int
        Total number of block placements (including air).
    """

    here = here.block()
    there = there.block()

    # check that the ends are in the same plane
    plane = here.coplane(there)
    if plane not in {'xy', 'yz'}:
        raise ValueError(
            f'Coordinates {here} and {there} are not in the xy or yz plane.')
    pointing = here.cardinal_direction(there, 'coord')
    facing = here.cardinal_direction(there)
    directions = {'north': '-z', 'south': 'z', 'east': 'x', 'west': '-x'}
    direction = directions[facing]
    axis = direction[-1]
    across = 'x' if axis == 'z' else 'z'

    # calculate the number of sections, and leftover for the ends
    dist = here.distance(there) + 1  # endpoint inclusive
    n_sections = int(dist // section_lengths)
    leftover = int(dist % section_lengths)

    # top of the support column as a Coord centered on the start position
    column_top = here.copy()
    column_top.y -= (radius + 1)

    print(f'Will build subway line from {here} to {there}:')
    print(f'    traveling {dist} blocks {facing}, in {n_sections} sections '
            f'of {section_lengths} blocks,\n'
            f'    with {leftover} blocks split between the ends.')
    if calcs_only:
        return plane, pointing, facing, dist, leftover, n_sections
    

    # define the different cross-sections of the subway tube
    cross_section = subway_section(radius)
    # add junction between sections
    junction = cross_section.copy()
    # add stone platform and add the gap section one beyond the end
    junction.loc[
        (np.abs(junction.x)>0) & (junction.y==-2), 'blocktype'] = blocktypes[98]
    # replace glass with stone and add ring/s before and after the gap
    ring = junction.copy()
    ring.loc[round(ring.radius) == radius, 'blocktype'] = blocktypes[1]
    # arch to place between support columns under each section
    arch = arch_df(section_lengths, 'STONE', dir=direction)

    on_ramp = int(leftover // 2)
    off_ramp = leftover - on_ramp

    # loop over subsections, sandwiched by on-ramp and off-ramp sections
    n_blocks = 0
    for section in range(n_sections + 2):

        midsection = False
        if section == 0:
            # first section is the leftover at the start
            section_length = on_ramp
            along = 0

        elif section == n_sections + 1:
            # last section is the leftover at the end
            section_length = off_ramp
            along = on_ramp + n_sections * section_lengths
        else:
            # regular section, with supports underneath
            midsection = True
            section_length = section_lengths
            along = on_ramp + (section - 1) * section_lengths

        if section_length == 0:
            continue  # no leftover section, skip

        # offset from the start position in the build direction
        if direction[0] == '-':
            along = -along
        along = Coord(0, 0, along) if axis == 'z' else Coord(along, 0, 0)

        # extrude the tube for this section
        added = extruder(cross_section, here + along, direction, section_length)
        n_blocks += added

        if section == 0:
            # add a ring at the entrance to hold the water in
            added = extruder(ring, here, axis, 1)
            n_blocks += added
            
        time.sleep(5 * delay_factor)

        if midsection or (n_sections > 0 and section == n_sections + 1):

            n_chunk = 0
            # add junction rings and support column at the start
            added = extruder(junction, here + along, axis, 1)
            n_chunk += added
            # replace glass with stone and add rings before and after the gap
            offset1 = Coord(0, 0, 1) if axis == 'z' else Coord(1, 0, 0)
            added = extruder(ring, here + along - offset1, axis, 1)
            n_chunk += added
            added = extruder(ring, here + along + offset1, axis, 1)
            n_chunk += added

            # add the support column at the start
            column = support(column_radius)
            column_pos = column_top + along
            # square slab at top, 2 * column_radius + 1 wide
            added = extruder(column, column_pos, "-y", 1)
            n_chunk += added
    
            # square below that, 2 * column_radius - 1 wide
            column = column[(np.abs(column.x) < column_radius) 
                            & (np.abs(column.y) < column_radius)]
            added = extruder(column, column_pos, "-y", 1)
            n_chunk += added
            # 4 pillars below, each column_radius - 2 wide, 1 block gap in middle
            column_pos.y -= 1
            column = column[(np.abs(column.x) < column_radius-1)
                            & (np.abs(column.y) < column_radius-1)]
            # should be smarter about this by checking height above the surface
            added = extruder(column, column_pos, "-y", max_support_height)
            n_chunk += added

            time.sleep(4 * delay_factor)
            n_blocks += n_chunk

        if midsection and n_sections > 0:
            # add the arches under this midsection, matched to the 4 columns
            transverse = sorted(column['x'].drop_duplicates())
            n_chunk = 0
            for x_offset in transverse:
                if x_offset == 0:
                    continue
                arch_pos = Coord(x_offset, 0, 0) if axis == 'z' \
                    else Coord(0, 0, x_offset)
                added = extruder(arch, column_top + along + arch_pos, across, 1)
                n_chunk += added

            time.sleep(3 * delay_factor)
            n_blocks += n_chunk
        
        # report progress
        if section == 0:
            print('Completed the on-ramp section')
        elif section < n_sections + 1:
            print(f'Finished midsection {section} of {n_sections}')
        else:
            print('Completed the off-ramp section')

    # add a ring at the end to hold the water in
    added = extruder(ring, there, axis, 1)
    n_blocks += added

    # put a minecart at the start of the subway section, on unpowered rail
    cart_pos = here + pointing + Coord(0, -1, 0)
    mc.setBlock(*(cart_pos - pointing), 'STONE', 'NORTH')
    mc.setBlock(*cart_pos, 'RAIL', 'NORTH')
    mc.runCommands('summon minecart ' + str(cart_pos)[1:-1])

    # also add block and unpowered rail at the end
    cart_pos = there - pointing + Coord(0, -1, 0)
    mc.setBlock(*cart_pos, 'RAIL', 'NORTH')
    mc.setBlock(*(cart_pos + pointing), 'STONE', 'NORTH')
    n_blocks += 4

    print(f'\nSubway complete!  Placed {n_blocks} blocks in total.')

    return n_blocks

### Actually lay "subway" track.
- Will start at your position, to a specified endpoint (`there`)
- Works best with `radius = 4`
- There are `time.sleep()` calls to avoid server overload, but may be 
  insufficient if you haven't passed through the area it is building.
- Should be based on numbers of block placed, but... not yet

#### Teleport player to a cool starting point abd build
#### Take several minutes to complete; please be patient.

In [33]:
# # random hillside to cave (takes ~ 5 minutes)
# south_end = 567
# north_end = 207
# xy = (-52, 103)

# # random hillside to village
# south_end = 1257.5
# north_end = 141.5
# xy = (-178.5, 69)

# village to hillside above village
xy = (-162.5, 87)
south_end = 1095.5
north_end = 212.5


# teleport to the start and adjust the start position
observer = Coord(*xy, south_end).midblock()
teleport(observer)

pos = observer.copy()
pos.y += 1  # track at the same block level as the player
pos.z -= 3  # two blocks north of the player (note that junction will be closer)

_ = build_subway_line(
    here=pos.midblock(), 
    there=Coord(pos.x, pos.y, north_end).midblock(),
    section_lengths=30,
    calcs_only=False,
)

Teleported Acaimo from [-29.5 92. 428.5] to [-162.5  87. 1095.5]
Will build subway line from [-163   88 1092] to [-163   88  212]:
    traveling 881.0 blocks north, in 29 sections of 30 blocks,
    with 11 blocks split between the ends.
Completed the on-ramp section
Finished midsection 1 of 29
Finished midsection 2 of 29
Finished midsection 3 of 29
Finished midsection 4 of 29
Finished midsection 5 of 29
Finished midsection 6 of 29
Finished midsection 7 of 29
Finished midsection 8 of 29
Finished midsection 9 of 29
Finished midsection 10 of 29
Finished midsection 11 of 29
Finished midsection 12 of 29
Finished midsection 13 of 29
Finished midsection 14 of 29
Finished midsection 15 of 29
Finished midsection 16 of 29
Finished midsection 17 of 29
Finished midsection 18 of 29
Finished midsection 19 of 29
Finished midsection 20 of 29
Finished midsection 21 of 29
Finished midsection 22 of 29
Finished midsection 23 of 29
Finished midsection 24 of 29
Finished midsection 25 of 29
Finished midsecti

### Build From Wherever!
#### build subway line from player's position, in direction they are facing

In [35]:

observer = Coord(mc.player.getTilePos()).block()
pointing = Vec3D(mc.player.getDirection()).cardinal_direction('coord')
print(observer)
   
n_sections = 3
section_length = 30
on_off_ramps = 20
pos = observer.copy()
pos.y += 1  # track at the same block level as the player
pos += pointing * 3  # two blocks in front of the player

there = pos + pointing * (n_sections * section_length + on_off_ramps)
_ = build_subway_line(
    here=pos, 
    there=there,
    section_lengths=section_length,
    delay_factor=2,
    calcs_only=False
)

[-45 118 546]
Will build subway line from [-42 119 546] to [ 68 119 546]:
    traveling 111.0 blocks east, in 3 sections of 30 blocks,
    with 21 blocks split between the ends.
Completed the on-ramp section
Finished midsection 1 of 3
Finished midsection 2 of 3
Finished midsection 3 of 3
Completed the off-ramp section

Subway complete!  Placed 12807 blocks in total.


### Search for nearby biomes
I was thinking of teleporting to each biome and trying to randomly sample
all the blocktypes in the game... but I think this would take a long time.

In [ ]:
biomes = sorted({m.value for _, m in Biome.__members__.items()})
biomes_at = mc.runCommands([f'locate biome {b}' for b in biomes], 1.0)
biomes_at = {b: v for b, v in zip(biomes, biomes_at)}
biomes_at

------------------
# The `rcon`, `mcipc`, `mcwb` and `mciwb` Libraries

This is just some stuff I messed around with before modifying `FruitJuice` 
and `pyncraft` modules.


------------------

## Use RCON For Commands Via The mcipc Client
- Allows you to run any command you can do from the command line
- Powerful, but limited-- can't get entity UIDs or Block state (data)

In [ ]:
def nbt_to_dict(nbt: str) -> dict:
    """
    Convert NBT data to a dictionary using json.loads.

    This mostly works, but not always.
    """
    if nbt.endswith('...'):
        raise ValueError(
            f"NBT data is truncated (ends with '...'); can't parse:\n{nbt}")

    match = re.search(r'^([A-Za-z0-9]+) has the following ([A-Za-z_]+) data: (.*)$', nbt)
    if not match:
        raise ValueError(f"Failed to parse NBT string:\n{nbt}")
    
    what = match.group(1)
    data_type = match.group(2)
    data = match.group(3)

    pre = '([ {\[])'
    term = '([ ,}\]])'
    # convert 0b and 1b to False and True
    data = re.sub(f'{pre}0b{term}', r'\1false\2', data)
    data = re.sub(f'{pre}1b{term}', r'\1true\2', data)
    # numeric types : remove "s", "l", "d" and "f" suffixes
    data = re.sub(f'{pre}(-?[0-9]+)[sl]?{term}', r'\1\2\3', data)
    data = re.sub(f'{pre}(-?[0-9\.]+)[df]{term}', r'\1\2\3', data)
    # convert other strings ending in a letter to string
    data = re.sub(f'{pre}([0-9]+[a-z]){term}', r'\1"\2"\3', data)
    # double quotes around the strings
    prefix = '([\[{ ])'
    data = re.sub(f'{prefix}([A-Z_a-z]+): ', r'\1"\2": ', data)
    # remove type indicator prefix from lists
    data = re.sub(f'\[[ILFDB]; ', r'[', data)

    # to avoid hitting string length limit, extract embedded lists first
    lists = re.findall('"([A-Z_a-z]+)": (\[[^\]]+\])', data)
    list_dict = {k: v for k, v in lists}
    updater = {}
    for k, v in list_dict.items():
        data = data.replace(v, '[]')
        d = json.loads('{"' + k + '": ' + v + '}')
        updater[k] = d[k]
        
    d = json.loads(data)
    d.update(updater)

    return what, data_type, {k: d[k] for k in sorted(d.keys())}

In [ ]:

# interact via the RCON port using the mcipc Client
# I've been running this in the debugger with a breakpoint set a few lines in.
# The client seems to only work in a context manager
with Client('localhost', rcon_port, passwd=rcon_pw) as client:
    seed = client.seed
    players = dict(client.list())

    here = (442, 73, -1060)
    client.setblock(pos=here, block='stone_stairs[facing=north,half=top]')
    
    client.run('setblock', '442', '73', '-1062', 'stone_stairs[facing=north,half=top]')#, 'half', 'top')
    # # these examples from the docs don't work:
    # mansion = client.locate('mansion')
    # badlands = client.locatebiome('badlands')
    client.run('tp', 'Acaimo', *here)
    client.run('effect', 'give', 'Acaimo', 'night_vision', 'infinite', 255, True)

    # but these will:
    mansion = client.run('locate', 'structure', 'mansion')
    badlands = client.run('locate', 'biome', 'badlands')

    # when run with Paper/FruitJuice, output is truncated to 200 characters
    nbt = client.data.get(entity='Acaimo')  
    # nbt = client.data.get(block=Vec3(193, 71, 61)) # blocks have states, not NBT data
    what, dtype, data = nbt_to_dict(nbt)
    print(f'{dtype}: "{what}"') 


    # print(client.xp()('Acaimo', 1000))

    
print(seed)   
print(players)
display({k: v for k, v in data.items() if k in ['pos', 'Pos', 'Rotation', 'UUID']})      

## Playing with `mciwb` MineCraftServer
- I got this to work sometimes, but then it wouldn't reconnect.
- The package has a bunch of code to download and control the docker server
  as a class.  But you still have to install docker first.
- I think it is easier to just specify the server config in `docker-compose.yml`
- You can then start and stop the server like:
```
# to start the server:
> docker compose up # add -d flag for detached mode

# if not detached, use cntl-C to stop the server
# to remove the server (if you want to reconfigure it)
> docker compose down
```

In [ ]:
# # attempt to get the mciwb MineCraftServer to start up; then connect with Client
# pw = 'aspergers-with-cheese'
# server_name = 'mciwb-server'
# server_port = 20101
# rcon_port = server_port - 1
# server_folder = Path.home() / 'minecraft/mciwb-server'
# backup_folder = Path.home() / 'minecraft/mciwb-backup'
# world_type = 'normal'
# player_name = 'Acaimo'

# mc = MinecraftServer(
#     name=server_name,
#     rcon=rcon_port,
#     password=pw,
#     server_folder=server_folder,
#     world_type=world_type,
#     backup_folder=backup_folder,
# )

# print('\n    '.join(['MinecraftServer methods:'] + [
#     m for m in dir(mc) if not m.startswith('_') and isinstance(getattr(mc, m), Callable)]))

# print('\n    '.join(['MinecraftServer attributes:'] + [
#     m for m in dir(mc) if not m.startswith('_') and not isinstance(getattr(mc, m), Callable)]))

# mc.create()

## Try using the MCIWB client
- this works, but no real advantage over mcipc

In [ ]:
from mciwb.threads import get_client, set_client, get_thread_name

 
set_client(Client('localhost', rcon_port, passwd=pw))
# client = get_client()

In [ ]:
# as with the MCIPC Client, must use the context manager
with get_client() as client:
    here = (50, 76, -160)
    client.setblock(Vec3(*here), 'glass')

# client is essentially the same as the MCIPC Client